# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
- 이름: 황수아
- GitHub ID: totoriri605-oss
- 작성일: 2026.10.01
- 최종 Notebook URL: https://github.com/totoriri605-oss/llm-data-analysis-course/blob/main/chapter10/chapter10.ipynb

> **먼저 밝혀 둘 것 (공식 실습 코드/데이터 사용 중 확인한 점)**
> - 강의 공개 저장소의 `src/classification.py`, `scripts/*`, `data/raw/*`를 그대로 복사해서 썼다. (내 프로젝트의 Chapter 09 데이터와 섞이지 않게 `data/ch10/` 아래에 따로 뒀다.)
> - 그런데 공식 raw 데이터를 그대로 넣고 `run_classification_analysis.py`를 돌리니 `signup_date가 order_date보다 늦은 주문이 있습니다: 45건` 에러가 났다.
>   (가입일보다 주문일이 앞선 주문이 전체 300건 중 56건이고, 그중 completed/cancelled는 45건)
> - 날짜를 고치거나 몰래 지우는 대신, `scripts/prepare_ch10_data.py`에 **"가입일보다 주문일이 앞선 주문(+그 주문상세)을 제외하고 목록을 `reports/ch10_excluded_pre_signup_orders_internal.csv`에 저장"** 하는 단계를 내가 추가했다.
> - 그래서 다른 학생의 결과와 행 수/지표가 다를 수 있다. 최종 모델링 행 수는 203건이다.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

print(sys.executable)
print(Path.cwd())

project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
report_dir = project_root / "reports"
print("프로젝트 루트:", project_root)
print("입력 파일:", sorted(p.name for p in (project_root / "data" / "ch10" / "processed").glob("*_clean.csv")))

C:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
C:\dev\llm-data-analysis-course\chapter10
프로젝트 루트: C:\dev\llm-data-analysis-course
입력 파일: ['customers_clean.csv', 'order_items_clean.csv', 'orders_clean.csv', 'products_clean.csv']


## 1. 분류 문제 정의
- 타깃: `is_cancelled`
- 0의 의미: completed (완료 주문)
- 1의 의미: cancelled (취소 주문)
- 제외한 상태: refunded 및 기타 상태 (41건)
- 예측 단위: 주문 1건
- 예측 시점: 주문 생성 직후
- 예측 시점에 사용할 수 있는 정보: 고객 나이/성별/지역, 가입 후 경과일(`days_since_signup`), 결제수단, 주문 월/요일, 주문 상품 수(`item_count`), 총 수량(`total_quantity`), 주문 금액(`order_amount`)
- 예측 이후에만 알 수 있어 제외한 정보: `order_status`, `is_cancelled`, `cancel_reason`, `cancelled_at`(취소 이후 정보), 식별자(`order_id`, `customer_id`, `product_id`)

### 나의 해석과 판단
주문 상태가 completed/cancelled인 주문만 쓰면 "이 주문이 취소될까?"를 0/1로 딱 정의할 수 있다.
refunded는 환불이라 취소와 성격이 다르고(이미 결제/배송이 끝난 뒤일 수도 있음) 이번 타깃 범위에 섞으면 의미가 흐려져서 뺐다.
예측 시점을 "주문 생성 직후"로 잡은 이유는, 그 시점에 이미 알 수 있는 정보만 입력으로 쓰는 연습을 하기 위해서다.
주문 금액이나 상품 수는 주문을 만들 때 정해져 있으니 쓸 수 있다고 가정했다. (교육용 가정이라 실제 업무에서는 확인이 필요하다.)

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`, `days_since_signup`

### 사용한 범주형 Feature
`gender`, `city`, `payment_method`

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보 (cancel_reason, cancelled_at)
```

(금지 feature와 실제 입력의 겹침: 0개, 자동 Validation `forbidden_feature_overlap` PASS)

### 나의 해석과 판단
컬럼이 파일에 있다고 해서 주문이 만들어지는 순간에 알 수 있는 값은 아니다. 예를 들어 `order_status`는 파일에는 있지만 주문이 끝난 뒤에야 정해지는 값이고,
타깃(`is_cancelled`)을 바로 알려주는 값이라 넣으면 거의 정답을 보고 푸는 셈이 된다. `cancel_reason`, `cancelled_at`도 취소가 일어난 뒤에 생기는 정보라 같은 이유로 뺐다.
ID들은 숫자여도 크기에 의미가 없고, 특정 주문/고객을 외우는 식으로 학습할 위험이 있어서 뺐다.

---

In [2]:
audit = pd.read_csv(report_dir / "ch10_feature_audit.csv")
display(audit)

,column,selected,role,reason
0,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,item_count,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,total_quantity,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,order_amount,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,days_since_signup,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
7,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
8,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
9,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정


## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: **0건** (`ch10_data_quality_checks.csv`의 `line_total_mismatch`)
- 주문 특징 `order_id` 중복 건수: **0건** (주문 특징은 주문당 1행, 아래 코드 셀에서 직접 확인)

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 203 | 203 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 203 | 203 | 0 | PASS |

### 결과 관찰
- 두 병합 모두 행 수가 그대로(203행)이고 미매칭이 0건이다.
- 모델링 전 단계에서 주문 300건 중 가입일보다 주문일이 앞선 56건을 제외했고(위 안내 참고), 그다음 refunded 41건이 타깃 범위에서 빠져 최종 203건이 됐다.
- 제외 후 `negative_days_since_signup`는 0건이다.

### 나의 해석과 판단
미매칭이 있을 때 원인 확인 없이 0으로 채우면(`fillna(0)`) 실제로는 "주문상세가 없는 주문"이나 "고객 정보가 없는 주문"인데 "금액이 0원인 주문"처럼 보이게 된다.
그러면 문제가 있다는 사실이 숨겨지고, 모델이 이 가짜 0을 진짜 패턴처럼 학습할 수도 있다.
그래서 미매칭이 있으면 에러로 멈추고 원인부터 확인하는 게 맞다. (이번 가입일 문제도 같은 생각으로 몰래 고치지 않고 제외 목록을 남겼다.)

---

In [3]:
from src.classification import build_order_item_features

items = pd.read_csv(project_root / "data" / "ch10" / "processed" / "order_items_clean.csv")
features = build_order_item_features(items)
print("주문 특징 행 수:", len(features), "/ order_id 중복:", features["order_id"].duplicated().sum())
mismatch = ((items["quantity"] * items["unit_price"] - items["line_total"]).abs() > 1e-6).sum()
print("line_total 불일치:", mismatch)
display(pd.read_csv(report_dir / "ch10_merge_checks.csv"))
display(pd.read_csv(report_dir / "ch10_data_quality_checks.csv"))

주문 특징 행 수: 244 / order_id 중복: 0
line_total 불일치: 0


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,203,203,True,0,PASS
1,customer_id → customers,203,203,True,0,PASS


,check_item,count
0,binary_target_rows,203
1,completed_rows,149
2,cancelled_rows,54
3,excluded_status_rows,41
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,149
9,status_scope:cancelled,54


## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 149 | 73.4% |
| 1 | cancelled | 54 | 26.6% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
- completed 149건, cancelled 54건으로 cancelled가 약 27%다. 불균형이 있는 편이다(완전히 극단적이진 않음).
- 모든 주문을 completed라고 찍기만 해도 accuracy가 약 73.4%가 된다.

### 나의 해석과 판단
클래스가 한쪽으로 쏠려 있으면 accuracy가 높아도 소수 클래스(cancelled)를 하나도 못 맞힐 수 있다.
그래서 accuracy만 보면 안 되고 cancelled를 얼마나 찾았는지(recall), 찾았다고 한 것 중 맞은 비율(precision)을 같이 봐야 한다.

---

In [4]:
display(pd.read_csv(report_dir / "ch10_target_distribution.csv"))

,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.732 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.634 | 0.400 | 0.727 | 0.516 |
| Random Forest | 0.659 | 0.286 | 0.182 | 0.222 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: **Logistic Regression** (비베이스라인 후보 중 F1 → Recall → Precision 순으로 비교)

### 결과 관찰
- Dummy는 accuracy가 0.732로 가장 높지만 precision/recall/F1이 모두 0이다. 취소 주문을 하나도 못 찾았다는 뜻이다.
- Logistic Regression은 accuracy는 0.634로 더 낮지만 recall 0.727, F1 0.516이다.
- Random Forest는 recall이 0.182, F1이 0.222로 낮다.

### 나의 해석과 판단
Dummy보다 accuracy는 낮은데 모델을 더 좋다고 보는 이유는, Dummy는 취소 주문을 전혀 못 찾기 때문이다.
선택한 Logistic Regression은 취소 주문의 약 73%를 찾아냈다. 다만 찾았다고 한 것 중 실제 취소는 40%뿐이라 오탐(FP)이 많다.

### 한계와 추가 확인 사항
Validation이 41건이고 그중 cancelled는 11건뿐이라, 지표가 한두 건에 크게 흔들릴 수 있다.
모델 비교는 threshold 0.5 기준이고, 다른 threshold에서는 순위가 달라질 수도 있다.

---

In [5]:
display(pd.read_csv(report_dir / "ch10_validation_model_comparison.csv").round(3))

,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.634,0.400,0.727,0.516
1,Random Forest,validation,0.659,0.286,0.182,0.222
2,Dummy Most Frequent,validation,0.732,0.000,0.000,0.000


## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 89 | 32 |
| Validation | 모델·Threshold 선택 | 30 | 11 |
| Test | 선택 이후 최종 평가 | 30 | 11 |

(세 split 모두 두 클래스가 있고 cancelled 비율이 약 26~27%로 비슷하다. 자동 Validation `all_splits_have_both_classes` PASS)

### 나의 해석과 판단
Test를 보고 모델이나 threshold를 고르면, 그 Test가 이미 선택에 쓰여서 "처음 보는 데이터"가 아니게 된다.
그러면 Test 점수는 실제보다 좋게 나온 값이 되고(선택 과정에 맞춰진 점수), 새 데이터에서의 성능을 믿을 수 없다. 그래서 선택은 Validation에서 끝내고 Test는 마지막에 한 번만 본다.

### 교육용 random split의 한계
이번에는 클래스 비율을 유지하는 무작위 분할(stratified random split)을 썼다. 실제로는 과거 주문으로 학습해서 앞으로 들어올 주문을 예측하니까,
운영 전에는 시간 순서로 나눈 평가(out-of-time 평가)를 추가해야 한다. 같은 고객의 주문이 여러 split에 섞였을 가능성도 확인해야 한다.

---

In [6]:
display(pd.read_csv(report_dir / "ch10_split_summary.csv"))

,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


## 7. Validation 성능과 우선 지표

(선택 모델 Logistic Regression, 선택 threshold 0.6 기준, `ch10_validation_threshold_metrics.csv`)

- Accuracy: 0.780
- Precision: 0.583
- Recall: 0.636
- F1: 0.609

### 내가 가장 중요하게 본 지표
Recall (그리고 Recall과 Precision의 균형을 보는 F1)

### 그 이유
취소될 주문을 미리 알아야 대응할 수 있다고 가정하면 취소 주문을 놓치지 않는 게 중요해서 recall을 먼저 봤다. 다만 recall만 높이면 오탐이 늘어서 F1도 같이 봤다.

> 실제 FP/FN 비용 정보는 없다. 위 판단은 "취소를 놓치는 비용이 더 클 것"이라는 **가정**이다.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.40 | 0.360 | 0.818 | 0.500 | 낮은 threshold: 취소를 많이 잡지만(recall 높음) 오탐도 많음 |
| 0.60 | 0.583 | 0.636 | 0.609 | 선택: Validation에서 F1이 가장 높은 지점 |
| 0.70 | 0.800 | 0.364 | 0.500 | 높은 threshold: 예측이 정확해지지만(precision 높음) 놓치는 취소가 늘어남 |

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
**0.6**

### 선택 이유
Validation에서 F1이 가장 높았고(0.609), precision 0.583과 recall 0.636이 어느 한쪽으로 크게 쏠리지 않았다.
threshold를 낮추면 recall은 올라가지만(0.20에서 1.0) precision이 떨어졌고, 높이면 precision은 올라가지만(0.70에서 0.8) recall이 0.364로 떨어졌다. (관찰)

### 나의 해석과 판단
Threshold도 모델처럼 데이터를 보고 고르는 값이라, Final Test에서 정하면 Test에 맞춰 튜닝한 셈이 된다. 그래서 Validation에서 정하고 고정한 뒤 Test는 한 번만 본다.
다만 Validation의 cancelled가 11건뿐이라 이 threshold가 우연히 좋아 보였을 가능성이 있다.

---

In [7]:
display(pd.read_csv(report_dir / "ch10_validation_threshold_metrics.csv").round(3))

,threshold,accuracy,precision,recall,f1
0,0.20,0.366,0.297,1.000,0.458
1,0.25,0.463,0.323,0.909,0.476
2,0.30,0.463,0.323,0.909,0.476
3,0.35,0.512,0.333,0.818,0.474
4,0.40,0.561,0.360,0.818,0.500
5,0.45,0.659,0.429,0.818,0.562
6,0.50,0.634,0.400,0.727,0.516
7,0.55,0.683,0.444,0.727,0.552
8,0.60,0.780,0.583,0.636,0.609
9,0.65,0.756,0.556,0.455,0.500


## 9. FP/FN 해석

- FP 의미: 실제로는 완료될 주문을 "취소 위험"이라고 예측한 경우
- FN 의미: 실제로는 취소될 주문을 "완료"라고 예측해서 놓친 경우
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소를 미리 대응하려는 목적이면 FN(취소를 놓침)이 더 부담스러울 수 있다. (가정)
- 그 판단에 추가로 필요한 업무 정보: 취소 한 건의 손실 비용, 취소 위험 주문에 개입(확인 연락, 재고 확보 조정 등)할 때 드는 비용, 개입이 실제로 취소를 줄이는지

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
Final Test 41건에서 TN 24, FP 6, FN 7, TP 4이다.
취소 11건 중 4건만 찾았고 7건을 놓쳤다. 반대로 완료 30건 중 6건을 취소 위험으로 잘못 잡았다.
이 모델로 취소 위험 주문에만 개입한다면, 개입 대상 10건 중 실제 취소는 4건뿐이라 대부분의 개입이 헛수고가 된다.

### 한계와 추가 확인 사항
실제 비용 정보가 없어서 FP와 FN 중 무엇이 더 큰 문제인지는 확정할 수 없다. Test가 41건(cancelled 11건)이라 숫자가 한두 건에 크게 흔들린다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.6
- Accuracy: 0.683
- Precision: 0.400
- Recall: 0.364
- F1: 0.381
- Validation 대비 변화: Accuracy 0.780 → 0.683, Precision 0.583 → 0.400, Recall 0.636 → 0.364, F1 0.609 → 0.381

### 결과 관찰
- 네 지표가 모두 Validation보다 낮아졌다. 특히 F1이 0.609에서 0.381로 크게 떨어졌다.
- Final Test accuracy(0.683)는 모든 주문을 completed로 찍는 Dummy의 accuracy(약 0.732, 혼동행렬에서 계산)보다 낮다.

### 나의 해석과 판단
Validation에서 좋아 보였던 성능이 Test에서 많이 내려갔고, 이는 선택한 threshold와 모델이 Validation 데이터에 우연히 잘 맞았거나(데이터가 작아서) 일반화가 잘 안 된다는 신호로 본다.
그래서 이 모델이 새 주문에서도 잘 맞는다고 말하기는 어렵다. 원인을 단정할 수는 없지만 데이터가 작고(Test 41건) cancelled가 적은 영향이 클 것 같다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

(Test 지표 파일의 `selection_status`가 `frozen_before_test`로 기록됨)

---

In [8]:
display(pd.read_csv(report_dir / "ch10_test_metrics.csv").round(3))
display(pd.read_csv(report_dir / "ch10_confusion_matrix.csv", index_col=0))

,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.6,frozen_before_test,0.683,0.4,0.364,0.381


,pred_completed,pred_cancelled
actual_completed,24,6
actual_cancelled,7,4


## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
- 공개 prediction 컬럼: ['record_id', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
- 위 금지 컬럼이 공개 prediction에 있는지 아래 셀에서 직접 확인했고, 없다. (자동 Validation `public_prediction_identifier_columns`: none / PASS)
- 식별자가 들어간 내부 파일(`*_internal.csv`)은 로컬에만 두고 `.gitignore`로 GitHub에 올리지 않았다.

### 나의 해석과 판단
오류 분석에는 어떤 주문을 틀렸는지 알아야 해서 원본 위치(`source_index`)나 ID가 필요하다. 하지만 공개 결과에 ID가 있으면 특정 주문이나 고객과 연결될 수 있다.
그래서 오류 분석용은 내부(Internal)로 두고, 공개용에는 `record_id`처럼 원본과 연결되지 않는 임시 번호만 남긴다.

---

In [9]:
public = pd.read_csv(report_dir / "ch10_classification_predictions.csv")
banned = {"source_index", "order_id", "customer_id", "product_id"}
print("공개 컬럼:", public.columns.tolist())
print("금지 컬럼 포함:", sorted(banned & set(public.columns)))
display(public.head(3))

공개 컬럼: ['record_id', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
금지 컬럼 포함: []


,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,0,0,0.569824,Logistic Regression,0.6
1,test_0002,0,0,0.576583,Logistic Regression,0.6
2,test_0003,0,0,0.328096,Logistic Regression,0.6


## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과

| check | value | status |
| --- | --- | --- |
| target_exactly_completed_cancelled | [0, 1] | PASS |
| forbidden_feature_overlap | 0 | PASS |
| merge_rows_and_matches | True | PASS |
| all_splits_have_both_classes | True | PASS |
| model_selected_on_validation | True | PASS |
| threshold_selected_on_validation | True | PASS |
| public_prediction_identifier_columns | none | PASS |

(총 7개 check 모두 PASS)

### 자동 PASS가 의미하지 않는 것
자동 Validation은 코드가 정한 약속(타깃 범위, 금지 feature, 병합, 선택 순서, 식별자 제거)을 지켰는지만 확인한다.
그래서 이 모델이 실제 운영에 적합한지, 특정 고객군에 불리하지 않은지(공정성), 시간이 지나도 성능이 유지되는지는 PASS로 알 수 없다.
이번 결과처럼 Validation은 전부 PASS인데 Final Test 성능은 낮을 수도 있다.

---

In [10]:
display(pd.read_csv(report_dir / "ch10_classification_validation.csv"))

,check,value,status
0,target_exactly_completed_cancelled,"[np.int64(0), np.int64(1)]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS


## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. Final Test F1이 0.381, recall이 0.364, precision이 0.400로 낮고, Final Test accuracy(0.683)가 모두 completed로 찍는 Dummy(약 0.732)보다 낮다.
2. Validation에서 고른 threshold의 F1이 0.609에서 Test 0.381로 크게 떨어졌고, Validation/Test의 cancelled가 각각 11건, 11건뿐이라 지표를 믿기 어렵다.
3. Random split이라 미래 주문 예측 성능을 확인하지 못했고, 가입일 오류 주문 56건을 제외한 데이터로 분석했다. 실제 FP/FN 비용도 모른다.

### 현재 모델의 가장 큰 위험
취소 주문 11건 중 7건을 놓치고, 취소 위험으로 잡은 10건 중 6건은 실제로는 완료되는 주문이다. 이 모델에 기대어 개입하면 놓치는 취소도 많고 헛수고도 많다.

### 다음 개선 우선순위
1. 시간 순서로 나눈 평가(out-of-time)와 더 많은 데이터로 다시 검증하고, 가입일 오류 데이터의 원인을 확인한다.
2. FP/FN 비용 정보를 받아서 비용 기준으로 threshold를 다시 정한다. (새 Validation 설계 필요, 이 Test는 다시 쓰지 않는다)
3. 주문 생성 시점에 실제로 알 수 있는 변수를 더 찾아본다. (사후 정보는 제외)

### 예측과 인과 구분
이번에 확인한 것은 "입력 변수 조합으로 취소 주문이 어느 정도 구분되는지"라는 예측 패턴이고, 그마저도 Test에서는 약했다.
어떤 변수가 취소의 **원인**인지(예: 주문 금액이 크면 취소한다, 특정 지역이라 취소한다)는 이 분석으로 말할 수 없다.

---

## 전체 재실행 확인

```powershell
python scripts/prepare_ch10_data.py
python scripts/run_classification_analysis.py
```

- 실행 성공 여부: 성공 (종료 코드 0, 0)
- 선택 모델 / Threshold: Logistic Regression / 0.6
- Validation 전체 PASS 여부: 예

In [11]:
for script in ["scripts/prepare_ch10_data.py", "scripts/run_classification_analysis.py"]:
    run = subprocess.run([sys.executable, script], cwd=project_root, capture_output=True, text=True, encoding="utf-8")
    print(script, "종료 코드:", run.returncode)

saved = pd.read_csv(report_dir / "ch10_test_metrics.csv").iloc[0]
print("저장된 Final Test: threshold", saved["threshold"], "/ F1", round(saved["f1"], 3), "/", saved["selection_status"])
print("Validation 전체 PASS:", (pd.read_csv(report_dir / "ch10_classification_validation.csv")["status"] == "PASS").all())

scripts/prepare_ch10_data.py 종료 코드: 0


scripts/run_classification_analysis.py 종료 코드: 0
저장된 Final Test: threshold 0.6 / F1 0.381 / frozen_before_test
Validation 전체 PASS: True


---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [ ] 최종 Notebook URL을 제출합니다. (제출은 직접)